# 06 — RQ3: recommending remix sources

**Question.** For a real remix, can we recommend the *source track* it builds on? For each remix event, we rank the true source among 50 candidate tracks and measure how high it lands, using MRR and Hits@1/5/10.

**Leakage discipline.**
- Every feature is computed as of the remix time `T`.
- Only tracks that existed before `T` can be candidates.
- The test events are all remixes from 2022 onward (5,672 events, 162 remixers). The training events are a sample of 12,000 earlier remixes.

**The choice of negatives decides the result, so we report three regimes.**
- **Random:** any earlier track. This is an easy task, because old tracks are trivially ruled out.
- **Recent:** the 1,500 newest uploads before the remix. This regime still leaks age information: true sources are typically about two years old (median 763 days), while these candidates are new.
- **Age-matched:** tracks posted within ±90 days of the **true source**. This removes the age shortcut and is the **primary, fair test**.

**Evaluation rules.**
- Co-sources of the same remix are never used as negatives.
- Tied scores are broken at random, so heuristics with many ties (such as popularity) get no free wins.
- All MRR values have bootstrap 95 % confidence intervals, and model-vs-baseline gaps use a paired bootstrap.
- One ranker is trained per regime, and every ranker is evaluated on every regime.

In [1]:
import numpy as np, pandas as pd, time
from bisect import bisect_left
from collections import defaultdict
from sklearn.ensemble import HistGradientBoostingClassifier
rng=np.random.default_rng(0); DAY=86400
nc=pd.read_csv("data/processed/nodes_clean.csv"); loc=pd.read_csv("data/processed/edges.csv"); loc=loc[loc.edge_type=="local"].copy()
CUT=pd.Timestamp("2022-01-01",tz="UTC").timestamp()
id2t=dict(zip(nc.upload_id,nc.date_unix.astype("int64"))); id2auth=dict(zip(nc.upload_id,nc.user_name))
id2ns=dict(zip(nc.upload_id,nc.n_sources))
def toks(s): return set() if pd.isna(s) else {x.strip() for x in str(s).replace(";",",").split(",") if x.strip()}
id2tags=dict(zip(nc.upload_id,nc.usertags.apply(toks)))
srt=np.argsort(nc.date_unix.astype("int64").values); allids_s=nc.upload_id.values[srt]; allt_s=nc.date_unix.astype("int64").values[srt]
parent_kids={p:np.sort(s.child_date_unix.values) for p,s in loc.groupby("parent_id")}
auth_times={a:np.sort(g.values) for a,g in nc.groupby("user_name").date_unix}
ca=loc.child_id.map(id2auth); given={a:np.sort(loc.child_date_unix[ca==a].values) for a in ca.unique()}
prof={}
for a,g in nc.groupby("user_name"):
    gg=g.sort_values("date_unix"); prof[a]=(gg.date_unix.astype("int64").values,[id2tags[u] for u in gg.upload_id])
rem_srcauth=defaultdict(lambda: defaultdict(list))
for c,p,cd in zip(loc.child_id,loc.parent_id,loc.child_date_unix): rem_srcauth[id2auth[c]][id2auth[p]].append(cd)
for a in rem_srcauth:
    for b in rem_srcauth[a]: rem_srcauth[a][b]=np.sort(rem_srcauth[a][b])
print("events:",len(loc),"| candidates:",len(nc))

events: 59186 | candidates: 51486


## 1. As-of features, candidate pools, and a leakage spot-check

In [2]:
import pickle
from sklearn.metrics import roc_auc_score
from sklearn.inspection import permutation_importance
def asof(a,T): return 0 if a is None else int(bisect_left(a,T))
def prof_tags(a,T):
    p=prof.get(a)
    if not p: return set()
    ts,tg=p; s=set()
    for i in range(bisect_left(ts,T)): s|=tg[i]
    return s
def used_auth(rem,sa,T):
    d=rem_srcauth.get(rem); return 0 if (not d or sa not in d) else int(bisect_left(d[sa],T)>0)
FCOLS=["src_prior_remixes","src_age_days","src_n_sources","src_self","rem_prior_uploads","rem_prior_given","tag_jaccard","used_src_author_before"]
def feats(rem,sid,T,pt):
    sset=id2tags.get(sid,set()); jac=(len(pt&sset)/len(pt|sset)) if (pt or sset) else 0.0
    return [asof(parent_kids.get(sid),T),(T-id2t[sid])/DAY,id2ns.get(sid,0),int(id2auth.get(sid)==rem),
            asof(auth_times.get(rem),T),asof(given.get(rem),T),jac,used_auth(rem,id2auth.get(sid),T)]
def cap(T): return bisect_left(allt_s,T)

# all true sources of each remix -> never sample a co-source as a "negative"
child_srcs=loc.groupby("child_id").parent_id.apply(set).to_dict()

def _draw(lo,hi,k,excl,r):
    out=set(); g=0
    while len(out)<k and hi-lo>1 and g<k*20:
        s=allids_s[r.integers(lo,hi)]; g+=1
        if s not in excl: out.add(s)
    return list(out)
def negs_random(T,true,excl,k,r):            # easy: any earlier track
    return _draw(0,cap(T),k,excl,r)
def negs_recent(T,true,excl,k,r,win=1500):   # old "fair": 1,500 newest uploads before T
    hi=cap(T); return _draw(max(0,hi-win),hi,k,excl,r)
AGE_WIN=90*DAY
def negs_agematched(T,true,excl,k,r):        # new fair: posted within ±90 days of the TRUE source
    ts=id2t[true]
    lo=bisect_left(allt_s,ts-AGE_WIN); hi=min(bisect_left(allt_s,ts+AGE_WIN),cap(T))
    return _draw(lo,hi,k,excl,r)
REGIMES={"random":negs_random,"recent":negs_recent,"agematched":negs_agematched}

ev=loc[["child_id","parent_id","child_date_unix"]].copy(); ev["rem"]=ev.child_id.map(id2auth)
_e=ev.iloc[len(ev)//2]; _r=np.random.default_rng(1)
for fn in REGIMES.values():
    ng=fn(_e.child_date_unix,_e.parent_id,child_srcs[_e.child_id],20,_r)
    assert all(id2t[s]<_e.child_date_unix for s in ng)
    assert not (set(ng)&child_srcs[_e.child_id])
print("leakage spot-check passed: all candidates predate the remix and none is a true source")

leakage spot-check passed: all candidates predate the remix and none is a true source


## 2. Train three rankers — one per negative regime

In [3]:
train_ev=ev[ev.child_date_unix<CUT].sample(12000,random_state=0)
test_ev=ev[ev.child_date_unix>=CUT].reset_index(drop=True)
def build(neg_fn,seed):
    r=np.random.default_rng(seed); X=[]; y=[]
    for c,rem,ps,T in zip(train_ev.child_id,train_ev.rem,train_ev.parent_id,train_ev.child_date_unix):
        pt=prof_tags(rem,T); X.append(feats(rem,ps,T,pt)); y.append(1)
        for s in neg_fn(T,ps,child_srcs[c],10,r): X.append(feats(rem,s,T,pt)); y.append(0)
    return HistGradientBoostingClassifier(max_depth=4,learning_rate=0.05,max_iter=300,random_state=0).fit(np.array(X),np.array(y))
t0=time.time()
MODELS={f"model({k})":build(fn,seed) for (k,fn),seed in zip(REGIMES.items(),[10,11,12])}
print(f"trained {len(MODELS)} models in {time.time()-t0:.0f}s | test events={len(test_ev)} | distinct test remixers={test_ev.rem.nunique()}")

trained 3 models in 12s | test events=5672 | distinct test remixers=162


### 2b. Freeze the evaluation candidate sets

The 50 negatives for each test event are drawn once per regime, with fixed seeds, and saved to `data/processed/rq3_eval_sets.pkl`. Notebook `07` evaluates on exactly these candidates, so the two notebooks are directly comparable.

In [4]:
N_NEG=50; EVAL={}
for (k,fn),seed in zip(REGIMES.items(),[20,21,22]):
    r=np.random.default_rng(seed); rows=[]
    for c,rem,ps,T in zip(test_ev.child_id,test_ev.rem,test_ev.parent_id,test_ev.child_date_unix):
        ng=fn(T,ps,child_srcs[c],N_NEG,r)
        if len(ng)>=5: rows.append((int(c),rem,int(ps),int(T),[int(x) for x in ng]))
    EVAL[k]=rows
    print(f"{k:11s} usable events={len(rows)}  mean candidates/event={np.mean([len(x[4]) for x in rows])+1:.1f}")
with open("data/processed/rq3_eval_sets.pkl","wb") as fh: pickle.dump(EVAL,fh)
print("saved data/processed/rq3_eval_sets.pkl")

random      usable events=5672  mean candidates/event=51.0
recent      usable events=5672  mean candidates/event=51.0
agematched  usable events=5672  mean candidates/event=51.0
saved data/processed/rq3_eval_sets.pkl


## 3. Cross-evaluation: methods × negative regimes

For each regime, the table shows MRR (with 95 % CI) and Hits@1/5/10 for:
- a random guess,
- two heuristics: **recency** (newer is better) and **popularity** (more prior remixes is better),
- the three trained rankers.

Random-guess MRR with 51 candidates is about 0.09. The **age-matched** block is the primary result.

In [5]:
def rank_of_true(s,r):
    """rank of candidate 0; ties broken uniformly at random"""
    s=np.asarray(s,float); greater=int((s[1:]>s[0]).sum()); ties=int((s[1:]==s[0]).sum())
    return 1+greater+int(r.integers(0,ties+1))
def score_all(rows,r):
    rk={k:[] for k in ["random guess","recency","popularity",*MODELS]}; pc={}
    for c,rem,ps,T,ng in rows:
        cands=[ps]+ng
        pt=pc.get((rem,T))
        if pt is None: pt=prof_tags(rem,T); pc[(rem,T)]=pt
        Xc=np.array([feats(rem,s,T,pt) for s in cands])
        sc={"random guess":np.zeros(len(cands)),"recency":-Xc[:,1],"popularity":Xc[:,0]}
        for m,mod in MODELS.items(): sc[m]=mod.predict_proba(Xc)[:,1]
        for m,s in sc.items(): rk[m].append(rank_of_true(s,r))
    return {m:np.array(v) for m,v in rk.items()}
def summarize(ranks,r,B=1000):
    n=len(next(iter(ranks.values()))); idx=r.integers(0,n,(B,n)); out={}
    for m,v in ranks.items():
        rr=1/v; boot=rr[idx].mean(1)
        out[m]=dict(MRR=rr.mean(),MRR_lo=np.percentile(boot,2.5),MRR_hi=np.percentile(boot,97.5),
                    H1=(v<=1).mean(),H5=(v<=5).mean(),H10=(v<=10).mean())
    return pd.DataFrame(out).T
def paired_diff(ranks,a,b,r,B=1000):
    d=1/ranks[a]-1/ranks[b]; idx=r.integers(0,len(d),(B,len(d))); bt=d[idx].mean(1)
    return d.mean(),np.percentile(bt,2.5),np.percentile(bt,97.5)
RANKS={}; TABLES={}
for k in REGIMES:
    RANKS[k]=score_all(EVAL[k],np.random.default_rng(30)); TABLES[k]=summarize(RANKS[k],np.random.default_rng(31))
    print(f"\n=== negatives: {k}  (events={len(EVAL[k])}) ===")
    print(TABLES[k].round(3).to_string())
    best=max(MODELS,key=lambda m:TABLES[k].loc[m,"MRR"])
    md,lo,hi=paired_diff(RANKS[k],best,"popularity",np.random.default_rng(32))
    print(f"paired MRR diff {best} - popularity: {md:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
pd.concat(TABLES,names=["negatives","method"]).round(4).to_csv("results_rq3_recommend.csv")
print("\nsaved results_rq3_recommend.csv")


=== negatives: random  (events=5672) ===
                     MRR  MRR_lo  MRR_hi     H1     H5    H10
random guess       0.090   0.086   0.094  0.020  0.099  0.197
recency            0.478   0.467   0.489  0.379  0.574  0.660
popularity         0.204   0.197   0.211  0.088  0.291  0.450
model(random)      0.558   0.548   0.569  0.435  0.697  0.809
model(recent)      0.165   0.158   0.172  0.079  0.225  0.334
model(agematched)  0.353   0.344   0.362  0.213  0.496  0.675
paired MRR diff model(random) - popularity: +0.354  95% CI [+0.341, +0.366]

=== negatives: recent  (events=5672) ===
                     MRR  MRR_lo  MRR_hi     H1     H5    H10
random guess       0.086   0.082   0.090  0.017  0.096  0.192
recency            0.202   0.194   0.210  0.120  0.271  0.321
popularity         0.311   0.301   0.320  0.204  0.395  0.522
model(random)      0.274   0.266   0.284  0.169  0.357  0.461
model(recent)      0.351   0.341   0.361  0.230  0.457  0.620
model(agematched)  0.371   0.362  

## 4. Figure and feature importance (age-matched regime)

Permutation importance is computed for the age-matched ranker on test events, as a pairwise task: the true source against 5 age-matched negatives. The pairwise AUC is 0.828.

In [6]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
Xt=[]; yt=[]
for c,rem,ps,T,ng in EVAL["agematched"]:
    pt=prof_tags(rem,T); Xt.append(feats(rem,ps,T,pt)); yt.append(1)
    for s in ng[:5]: Xt.append(feats(rem,s,T,pt)); yt.append(0)
Xt=np.array(Xt); yt=np.array(yt); mA=MODELS["model(agematched)"]
print("age-matched pairwise AUC:",round(roc_auc_score(yt,mA.predict_proba(Xt)[:,1]),3))
pi=permutation_importance(mA,Xt,yt,scoring="roc_auc",n_repeats=8,random_state=0)
imp=pd.Series(pi.importances_mean,index=FCOLS).sort_values(); print(imp.round(4).to_string())
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
meth=list(TABLES["random"].index); x=np.arange(len(meth)); w=.27
for j,(k,col) in enumerate(zip(REGIMES,["#4c72b0","#dd8452","#c44e52"])):
    t=TABLES[k]
    ax[0].bar(x+(j-1)*w,t.MRR,w,yerr=[t.MRR-t.MRR_lo,t.MRR_hi-t.MRR],capsize=2,label=f"{k} negatives",color=col)
ax[0].set_xticks(x); ax[0].set_xticklabels(meth,rotation=20,ha="right",fontsize=8)
ax[0].set_ylabel("MRR (95% CI)"); ax[0].set_title("Source recommendation by negative regime"); ax[0].legend(fontsize=8)
ax[1].barh(imp.index,imp.values,color="#55a868"); ax[1].set_title("Feature importance, age-matched (ΔAUC)")
plt.tight_layout(); plt.savefig("fig_06_rq3_recommend.png",dpi=130); print("saved fig_06_rq3_recommend.png")    

age-matched pairwise AUC: 0.828
rem_prior_uploads         0.0030
tag_jaccard               0.0066
src_self                  0.0076
rem_prior_given           0.0081
used_src_author_before    0.0465
src_n_sources             0.0474
src_age_days              0.0551
src_prior_remixes         0.0847
saved fig_06_rq3_recommend.png


## 5. Findings

**Remix sources are recommendable well above chance** (random-guess MRR ≈ 0.09).

**Headline result (age-matched test, primary).** A learned ranker clearly beats the best heuristic, popularity:

| Method | MRR [95 % CI] | Hits@10 |
|---|---|---|
| Age-matched ranker | **0.345** [0.336, 0.353] | 0.67 |
| Popularity | 0.252 [0.245, 0.261] | 0.51 |
| Recency | 0.104 | — |

The paired difference between the ranker and popularity is **+0.092 [+0.084, +0.100]**. Recency is useless once age is matched.

**Results under the other regimes.**
- **Random negatives:** recency alone is strong (MRR 0.478), and the random-trained ranker reaches 0.558 (Hits@1 0.44). The task is easy, because old tracks are trivially ruled out.
- **Recent negatives:** the ranker again beats popularity (0.371 vs 0.311; +0.061 [+0.051, +0.071]).
- **Generalisation:** the ranker trained on age-matched negatives performs well in all three regimes (0.353 / 0.371 / 0.345). Rankers trained on random or recent negatives do much worse outside their own regime.

**What drives the ranker (age-matched).** In order of importance:
1. the source's prior remix count (0.085),
2. its age (0.055),
3. how many sources the source itself has (0.047),
4. whether the remixer has used that author before (0.046).

Tag similarity (0.007) and the remixer's own activity add little. Remixers choose sources that are already proven (true sources had 3.7 prior remixes on average vs 0.8 for age-matched alternatives), original rather than derivative, and by authors whose material they have used before (73.5 % vs 46.9 %), far more than material that matches their genre tags.

**Correction to an earlier version.** An earlier version reported that the ranker only *tied* popularity. That came from arbitrary tie-breaking, which favoured popularity, and from non-age-matched negatives.

**Limits.**
1. **Negative regime.** Absolute values depend on the negative regime, which is why all three are reported.
2. **Existing users only.** Only 162 remixers are active in the test window, so this is recommendation for existing active users, not for newcomers with no history.
3. **Split events.** Multi-source remixes are split into separate (source, remix) events.
4. **Observed choices only.** Evaluation uses remixes that actually happened, so it measures ranking of observed choices rather than live recommendation impact.